# thui-rank-v0 — B61 smoke: frame-change prior as a VETO over the LLM's proposals

**Infrastructure smoke, not a scoring run.** `thui-v3-0` (the B48 build: thui-v1-1 + yield 180, the standing-best chassis) byte-for-byte except cells 12 and 14.
Cell 12 wraps the analyzer so every action the LLM proposes is scored by a small CNN trained
online on this game's own executed actions; a proposal predicted inert (p < 0.15, after ≥ 20
observations, ≤ 2 vetoes per step) is answered with the harness's own invalid-action payload and
the LLM re-picks. **The prior never issues an action.** Cell 14 filters to tr87 / sk48 / sc25 at
900 s each. Numbers are meaningless and must never be quoted. Design: `notes/B61-prior-as-ranker-design.md`.

**Smoke variant (2026-09-04):** arming threshold lowered to **5** observations (design: 20) so the veto path is exercised at all inside 900 s -- v0 never armed. Proves the PATH, not the lever.

Solver credit: Tufa Labs (Harold Bessis, Jeroen Cottaar, Isaiah Pressman, Andries Smit,
Michal Tesnar, Stefano Viel) — executed unmodified from their attached dataset. This is a
Knowless Crew / Thuitanium fork; none of their scores are ours.


## 1. Environment and submission mode

Detect whether this is a real competition rerun (which minimises diagnostics), set the
framework's environment flags, and put the CUDA libraries on the linker path.

In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# In submission, disable the periodic JSON/HTML diagnostics writes and per-frame logging.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")

## 2. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment
has no internet).

In [ ]:
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

## 3. Locate the source bundle

Find the uploaded TAAF source dataset by its marker file, and record where Kaggle mounted
every attached input so setup commands and the solver can find them.

In [ ]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["jakobbrggen/taaf-kaggle-source-anim-20260807-anim", "driessmit1/arc3-vllm-h100-wheelhouse-v3", "jakobbrggen/qwen3-8-27b-fp8-hf-snapshot"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir() -> Path:
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        return marker.parent
    raise RuntimeError("TAAF source bundle not found under /kaggle/input.")


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir()
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

## 4. Import the bundled source and run solver setup

Put the snapshotted repositories on the path (this process and any child processes), then run
the solver's setup commands — installing wheels, fetching model weights, and so on.

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# Solver setup commands (wheels, vLLM server startup, ...) run before the benchmark loads.
env = _command_env()
for command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    # duckv10: model swap only (R12 seam). NO output cap - v9 proved a 768-token ceiling
    # truncates the tool call that carries the action itself.
    command = (
        command
        .replace("MODEL_OWNER = 'driessmit1'", "MODEL_OWNER = 'jakobbrggen'")
        .replace(
            "MODEL_SLUG = 'vrfai-qwen3-6-27b-fp8-hf-snapshot'",
            "MODEL_SLUG = 'qwen3-8-27b-fp8-hf-snapshot'",
        )
        .replace(
            "SERVED_MODEL_NAME = 'vrfai/Qwen3.6-27B-FP8'",
            "SERVED_MODEL_NAME = 'vrfai/Qwen3.8-27B-FP8'",
        )
        .replace(
            "    'LOCAL_ANALYZER_TEMPERATURE':",
            "    'LOCAL_ANALYZER_SEED': '20260825',\n    'LOCAL_ANALYZER_TEMPERATURE':",
        )
        .replace(
            "'LOCAL_ANALYZER_YIELD_SECONDS': '60'",
            "'LOCAL_ANALYZER_YIELD_SECONDS': '180'",
        )
    )
    assert "vrfai-qwen3-6-27b-fp8-hf-snapshot" not in command, "duckv10: model slug rewrite missed"
    assert "Qwen3.6-27B-FP8" not in command, "duckv10: served-name rewrite missed"
    assert "'LOCAL_ANALYZER_MAX_OUTPUT': '0'" in command, "duckv10: output must stay UNCAPPED"
    # thui-v1-1 TEETH, in-kernel, before the benchmark starts. The seed IS this
    # build: if the anchor moved in a newer bundle the replace is a silent no-op and the
    # run would score normally while measuring nothing.
    assert "'LOCAL_ANALYZER_SEED': '20260825'" in command, (
        "thui-v1-1 TEETH FAIL: seed injection missed -- the setup_env anchor "
        "\"    'LOCAL_ANALYZER_TEMPERATURE':\" is not in this bundle's setup command"
    )
    assert command.count("'LOCAL_ANALYZER_SEED'") == 1, (
        "thui-v1-1 TEETH FAIL: seed key injected more than once"
    )
    assert "'LOCAL_ANALYZER_TEMPERATURE': '0.6'" in command, (
        "thui-v1-1 TEETH FAIL: temperature is not 0.6 -- this arm must not touch it"
    )
    assert "'MULTIMODAL_UPSCALE': '4'" in command, (
        "thui-v1-1 TEETH FAIL: upscale must stay 4 (v10 exact, B23 measured in-noise)"
    )
        # thui-v3-0 TEETH, in-kernel, before the benchmark starts. The YIELD value IS this
    # build: if the anchor moved in a newer bundle the replace is a silent no-op and the run
    # would score normally while measuring nothing (the duckv25 shape).
    assert "'LOCAL_ANALYZER_YIELD_SECONDS': '180'" in command, (
        "thui-v3-0 TEETH FAIL: yield injection missed -- "
        "\"'LOCAL_ANALYZER_YIELD_SECONDS': '60'\" is not in this bundle's setup command"
    )
    assert "'LOCAL_ANALYZER_YIELD_SECONDS': '60'" not in command, (
        "thui-v3-0 TEETH FAIL: the old 60 s value survives -- two values would race"
    )
    assert command.count("'LOCAL_ANALYZER_YIELD_SECONDS'") == 1, (
        "thui-v3-0 TEETH FAIL: yield key present more than once"
    )
    assert "'LOCAL_ANALYZER_TOOL_STEPS': '0'" in command, (
        "thui-v3-0 TEETH FAIL: TOOL_STEPS must stay 0 -- B47 measured a cap inert and this "
        "arm must not cross that variable"
    )
    print("thui-v3-0: YIELD_SECONDS 60 -> 180 (B47/R44); seed=20260825, temperature untouched", flush=True)
    print(f"taaf.kaggle: setup command: {command}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    # Re-read in case the command persisted new env keys.
    env = _command_env()
    os.environ.update(env)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)

## 5. Load the benchmark

Unpickle the deployment target and the benchmark, stamping the real submission state onto the
target and pointing the benchmark's outputs at the Kaggle working directory.

In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

## 6. Customization hook

Optional: tweak `bm`, `bm.games`, or `bm.solver` here before the run starts — the safe place
for one-off experiments once the deployed bundle has loaded.

In [ ]:
# === per-request usage probe: completion_tokens + finish_reason + wall time ===
# Runs after `bm`/`bm.solver` are unpickled (cell 10) and before `bm.run(...)` (cell 14),
# so every ToolAgent built during the run is already wrapped -- the analyzer is
# constructed per game inside bm.run (solver.py:1339 _make_analyzer).
#
# NOT `bm.solver.save_request_logs = True`: that flag writes the full message list on
# both the request and the response event (~150-215 MB for this run) and never writes
# `usage`, which is the one field that can place an output-token cap. duckv9 capped at
# 768 without that distribution and scored 0.22 (finish_reason `length` 704 against
# `tool_calls` 68). See docs/audit/2026-08-23-request-usage-probe.md.

from inference.agent import tool_agent

"""Per-request usage probe for the duck harness — the cell-12 payload.

WHY THIS EXISTS, and why it is not `save_request_logs=True`.

The harness already has a per-request logger. `HarnessSolver.save_request_logs`
(`solver.py:885`) flows to `ToolAgent` (`solver.py:1350`) and gates two
`_append_request_snapshot` calls (`tool_agent.py:2195` and `:2208`). Setting
`bm.solver.save_request_logs = True` in cell 14 works — the analyzer is built per game
inside `bm.run()` (`solver.py:1339 _make_analyzer`), so the flag is read after cell 12
and after cell 14's assignments.

It records the wrong thing for the question in front of us. `_append_request_snapshot`
(`tool_agent.py:929-965`) writes `messages` + `tools` + `finish_reason` + `analysis_step`
+ `action` + `request_index_within_turn`. It does **not** write `usage`, even though the
caller holds it — `result.usage` is passed to `_accumulate_usage_tokens` on the line
between the two snapshot calls (`:2207`) and then dropped. And it writes the full message
list on BOTH the request and the response event, so the response row costs ~60-90 KB to
carry one string. At 1,201 requests × 2 events that is roughly 150-215 MB to obtain 1,201
values of `finish_reason`.

The open question is where to place an output-token cap. `duckv9` capped at 768 and scored
0.22 (`finish_reason` `length` 704 against `tool_calls` 68 — the cap truncated the tool call
carrying the action). `duckv10` runs uncapped at a mean 2,211 output tokens per request. So
we know 91% of requests want more than 768 tokens and nothing about the shape above it.
Placing a safe cap needs the **distribution**, which means `completion_tokens` per request.
That is what this file records, at about 200 bytes per request instead of 150 KB.

⚠️ ANSWERED — and the 91% above is the PRIOR this probe was built to test, not a result.
Measured, the share is **68.4%**. The whole paragraph is left standing on purpose: it is the
reason the probe exists, and overwriting an assumption with its answer deletes the record of
what was assumed. Read `notes/R35-usage-distribution.md` for what the run actually found —
§"Two corrections to what was assumed when the probe was written" (`:50-51`) carries this
correction in the author's own words, and the section above it kills the cap idea
structurally: the distribution has **no fat tail**, so every cap that saves meaningfully cuts
the body. A cap at 8,192 saves 0.98% of output; at 12,288 it saves nothing at all.

WHAT IT RECORDS — one JSONL row per request, written next to the run's other artifacts
using the harness's own path convention (`<game>_usage.jsonl`):

    game, action, req_in_turn, wall_s, prompt_tokens, completion_tokens, total_tokens,
    finish_reason

`req_in_turn` is maintained here rather than read from the harness, because the counter the
harness keeps (`turn_count`) is a local in `analyze()`.

WHAT IT ANSWERS

  - the output-token distribution, hence a cap that trims the tail without truncating
    the median (`duckv9`'s failure mode);
  - requests per turn, and how that moves if `LOCAL_ANALYZER_YIELD_SECONDS` changes —
    the tie-breaker for whether raising the yield budget buys anything at all;
  - measured per-request wall time per game, against the 164 s the vLLM log implies.

CONSTRAINTS

  - stdlib only, no project imports beyond `inference.agent.tool_agent`, no top-level side
    effects — `install()` must be called explicitly. The notebook build step embeds this
    file's source into cell 12 and appends the `install(...)` call, the way duckmod splices
    `duck_tools.py`.
  - **fail-open, always.** This is instrument code riding on a run that costs a GPU slot.
    Every hook swallows its own exceptions and the probe disables itself rather than
    letting a logging bug end a game.
  - bounded: `MAX_ROWS_PER_GAME` caps output so a pathological loop cannot fill the disk.
"""

import json
import time

MAX_ROWS_PER_GAME = 2000

_state = {"installed": False, "orig_analyze": None, "orig_chat": None, "disabled": False}


def _usage_path(tool_agent, state_path):
    """Per-game JSONL path, using the harness's own artifact-location convention.

    Falls back to a sibling of the state file if the private resolver is not where we
    expect it — a bundle refresh may move it, and a moved resolver must not end a run.
    """
    try:
        return tool_agent._resolve_named_run_artifact(
            state_path, default_name="request_usage.jsonl", per_game_suffix="_usage.jsonl"
        )
    except Exception:
        return state_path.parent / (state_path.stem + "_usage.jsonl")


def _as_int(value):
    try:
        return max(0, int(value))
    except (TypeError, ValueError):
        return None


def install(tool_agent, *, max_rows=MAX_ROWS_PER_GAME):
    """Wrap ToolAgent.analyze and ToolAgent._chat_completion. Idempotent.

    `tool_agent` is the module (or any object exposing `ToolAgent`), passed in rather than
    imported so this file is testable against a stub without the harness installed.
    """
    if _state["installed"]:
        return False
    agent_cls = tool_agent.ToolAgent
    orig_analyze = agent_cls.analyze
    orig_chat = agent_cls._chat_completion

    def analyze(self, state_path, action_num, *args, **kwargs):
        # Establishes the per-turn context the _chat_completion wrapper stamps onto rows.
        # A turn is one analyze() call; requests within it are numbered from 1.
        try:
            self._probe_path = _usage_path(tool_agent, state_path)
            self._probe_action = action_num
            self._probe_req = 0
        except Exception:
            self._probe_path = None
        return orig_analyze(self, state_path, action_num, *args, **kwargs)

    def _chat_completion(self, messages, **kwargs):
        started = time.monotonic()
        try:
            result = orig_chat(self, messages, **kwargs)
        except Exception as exc:
            _record(self, started, None, type(exc).__name__, max_rows)
            raise
        _record(self, started, result, None, max_rows)
        return result

    agent_cls.analyze = analyze
    agent_cls._chat_completion = _chat_completion
    _state.update(installed=True, orig_analyze=orig_analyze, orig_chat=orig_chat)
    return True


def _record(agent, started, result, exception_name, max_rows):
    """Append one row. Never raises; on any failure the probe turns itself off."""
    if _state["disabled"]:
        return
    try:
        path = getattr(agent, "_probe_path", None)
        if path is None:
            return
        count = getattr(agent, "_probe_rows", 0)
        if count >= max_rows:
            return
        agent._probe_rows = count + 1
        agent._probe_req = getattr(agent, "_probe_req", 0) + 1

        usage = getattr(result, "usage", None) if result is not None else None
        usage = usage if isinstance(usage, dict) else {}
        row = {
            "game": path.stem[: -len("_usage")] if path.stem.endswith("_usage") else path.stem,
            "action": getattr(agent, "_probe_action", None),
            "req_in_turn": agent._probe_req,
            "wall_s": round(time.monotonic() - started, 3),
            "prompt_tokens": _as_int(usage.get("prompt_tokens")),
            "completion_tokens": _as_int(usage.get("completion_tokens")),
            "total_tokens": _as_int(usage.get("total_tokens")),
            "finish_reason": (
                "__exception__:" + exception_name
                if exception_name
                else str(getattr(result, "finish_reason", "") or "")
            ),
        }
        with open(path, "a", encoding="utf-8") as handle:
            handle.write(json.dumps(row, ensure_ascii=True))
            handle.write("\n")
    except Exception:
        # One broken write must not cost a game. Stop trying, keep playing.
        _state["disabled"] = True


def uninstall(tool_agent):
    """Restore the original methods. Exists for the tests, not for the notebook."""
    if not _state["installed"]:
        return False
    tool_agent.ToolAgent.analyze = _state["orig_analyze"]
    tool_agent.ToolAgent._chat_completion = _state["orig_chat"]
    _state.update(installed=False, orig_analyze=None, orig_chat=None, disabled=False)
    return True

_installed = install(tool_agent)
print(f"request-usage probe installed: {_installed}")


# === thui-rank-v0 (B61): frame-change prior as a VETO over the LLM's proposals ==========
# Seam: same class-level wrap of ToolAgent.analyze as B60. Every step_env call carries the
# LLM's proposal; the prior scores it BEFORE the engine sees it. The prior never issues an
# action -- a vetoed single proposal gets the harness's own invalid-action payload shape.
import math as _math
import hashlib as _hashlib
import random as _random
import torch as _torch
import torch.nn as _nn
import torch.nn.functional as _F
from inference.agent import tool_agent as _ta
from inference.framework import solver as _solver
from inference.agent.action_names import to_engine_action as _to_engine, to_model_actions as _to_model

_RANK_DEV = "cpu"
_RANK_TYPES = ("ACTION1", "ACTION2", "ACTION3", "ACTION4", "ACTION5")
_RANK_VETO_P = 0.15
_RANK_MIN_OBS = 5
_RANK_PER_STEP = 2
_RANK_STATS = {"proposals": 0, "scored": 0, "vetoes": 0, "batch_drops": 0, "false_veto_proxy": 0,
               "updates": 0, "observed": 0, "games": 0, "wrapper_errors": 0}


def _rank_onehot(grid):
    t = _torch.zeros(16, 64, 64)
    for r, row in enumerate(grid[:64]):
        for c, v in enumerate(row[:64]):
            t[max(0, min(15, int(v))), r, c] = 1.0
    return t


def _rank_sig(grid):
    return _hashlib.blake2b(repr(grid).encode(), digest_size=8).hexdigest()


class _RankNet(_nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = _nn.Conv2d(16, 32, 3, padding=1)
        self.c2 = _nn.Conv2d(32, 64, 3, padding=1)
        self.c3 = _nn.Conv2d(64, 128, 3, padding=1)
        self.c4 = _nn.Conv2d(128, 256, 3, padding=1)
        self.act_head = _nn.Linear(256, len(_RANK_TYPES))
        self.coord_head = _nn.Conv2d(256, 1, 1)

    def forward(self, x):
        h = _F.relu(self.c1(x)); h = _F.relu(self.c2(h))
        h = _F.relu(self.c3(h)); h = _F.relu(self.c4(h))
        return self.act_head(h.mean(dim=(2, 3))), self.coord_head(h).squeeze(1)


def _rank_key(name, data):
    if name == "ACTION6":
        return f"ACTION6:{int(data.get('y', 0))},{int(data.get('x', 0))}"
    return name


class _Ranker:
    """Per-game: observe executed actions -> train; score a proposal -> p(change)."""

    def __init__(self):
        self.net = _RankNet().to(_RANK_DEV)
        self.opt = _torch.optim.Adam(self.net.parameters(), lr=1e-3)
        self.buf = []
        self.seen = set()
        self.changed = set()          # (sig, key) observed to CHANGE the board -- never vetoed
        self.observed = 0
        self.vetoed_last = None       # (sig, key) of the last veto, for the false-veto proxy

    def observe(self, grid, payload):
        if not isinstance(payload, dict) or not payload.get("executed"):
            return
        name = str(payload.get("action_name") or "").upper()
        data = payload.get("action_data") or {}
        if name == "ACTION6":
            rc = (int(data.get("y", 0)), int(data.get("x", 0))); tidx = None
        elif name in _RANK_TYPES:
            rc = None; tidx = _RANK_TYPES.index(name)
        else:
            return
        sig = _rank_sig(grid); key = (sig, _rank_key(name, data))
        label = bool(payload.get("board_changed")) or int(payload.get("frame_count") or 0) > 1
        if label:
            self.changed.add(key)
        if self.vetoed_last == key and label:
            _RANK_STATS["false_veto_proxy"] += 1
            print(f"thui-rank: FALSE-VETO proxy -- {key[1]} re-proposed, executed, changed", flush=True)
        self.vetoed_last = None
        self.observed += 1; _RANK_STATS["observed"] += 1
        if key in self.seen:
            return
        self.seen.add(key)
        self.buf.append((_rank_onehot(grid), tidx, rc, 1.0 if label else 0.0))
        if len(self.buf) > 20000:
            self.buf = self.buf[-20000:]
        self._train(steps=2)

    def _train(self, steps):
        if len(self.buf) < 4:
            return
        self.net.train()
        for _ in range(steps):
            batch = _random.sample(self.buf, min(32, len(self.buf)))
            x = _torch.stack([b[0] for b in batch]).to(_RANK_DEV)
            act_logits, coord_map = self.net(x)
            loss = _torch.zeros((), device=_RANK_DEV); n = 0
            for i, (_, tidx, rc, y) in enumerate(batch):
                tgt = _torch.tensor(y, device=_RANK_DEV)
                if tidx is not None:
                    loss = loss + _F.binary_cross_entropy_with_logits(act_logits[i, tidx], tgt)
                else:
                    loss = loss + _F.binary_cross_entropy_with_logits(coord_map[i, rc[0], rc[1]], tgt)
                n += 1
            loss = loss / max(1, n)
            self.opt.zero_grad(); loss.backward(); self.opt.step()
            _RANK_STATS["updates"] += 1
            if _RANK_STATS["updates"] % 25 == 0:
                print(f"thui-rank: update n={_RANK_STATS['updates']} buf={len(self.buf)} loss={loss.item():.4f}", flush=True)
        assert _math.isfinite(loss.item()), "thui-rank: non-finite loss"

    def score(self, grid, name, data):
        """p(change) for one proposal, or None when the prior must not judge."""
        if self.observed < _RANK_MIN_OBS:
            return None
        key = (_rank_sig(grid), _rank_key(name, data))
        if key in self.changed:
            return None
        self.net.eval()
        with _torch.no_grad():
            act_logits, coord_map = self.net(_rank_onehot(grid).unsqueeze(0).to(_RANK_DEV))
        if name == "ACTION6":
            r = max(0, min(63, int(data.get("y", 0)))); c = max(0, min(63, int(data.get("x", 0))))
            return float(_torch.sigmoid(coord_map[0, r, c]).item())
        if name in _RANK_TYPES:
            return float(_torch.sigmoid(act_logits[0, _RANK_TYPES.index(name)]).item())
        return None


def _rank_proposals(arguments):
    """Return list of (name, data, raw) for the proposal, or None if not an action call."""
    if not isinstance(arguments, dict):
        return None
    if str(arguments.get("query") or "").strip():
        return None
    raws = arguments.get("actions") if arguments.get("actions") is not None else [arguments]
    if not isinstance(raws, list):
        return None
    out = []
    for raw in raws:
        if not isinstance(raw, dict):
            return None
        name = _to_engine(raw.get("action"))
        if not name:
            return None
        data = {}
        if name == "ACTION6":
            try:
                data = {"x": int(raw["col"]), "y": int(raw["row"])}
            except Exception:
                return None
        out.append((name, data, raw))
    return out


_rankers = {}
_orig_analyze = _ta.ToolAgent.analyze


def _rank_analyze(self, state_path, action_count, *args, valid_actions=None, step_env=None, **kwargs):
    session = getattr(step_env, "__self__", None)
    if session is None or not hasattr(session, "game"):
        return _orig_analyze(self, state_path, action_count, *args, valid_actions=valid_actions, step_env=step_env, **kwargs)
    ranker = _rankers.get(id(session))
    if ranker is None:
        ranker = _rankers[id(session)] = _Ranker()
        _RANK_STATS["games"] += 1
        print(f"thui-rank: new ranker for game #{_RANK_STATS['games']}", flush=True)
    vetoes_this_step = [0]

    def rec_step_env(arguments):
        grid = _solver._grid_from_state(session.game.current_state)
        try:
            props = _rank_proposals(arguments)
            if props and vetoes_this_step[0] < _RANK_PER_STEP:
                _RANK_STATS["proposals"] += len(props)
                scored = [(n, d, r, ranker.score(grid, n, d)) for (n, d, r) in props]
                _RANK_STATS["scored"] += sum(1 for s in scored if s[3] is not None)
                inert = [s for s in scored if s[3] is not None and s[3] < _RANK_VETO_P and s[0] != "RESET"]
                if inert:
                    if len(props) == 1:
                        n, d, r, p = inert[0]
                        vetoes_this_step[0] += 1; _RANK_STATS["vetoes"] += 1
                        ranker.vetoed_last = (_rank_sig(grid), _rank_key(n, d))
                        print(f"thui-rank: VETO #{_RANK_STATS['vetoes']} {_rank_key(n, d)} p={p:.3f} obs={ranker.observed} step_vetoes={vetoes_this_step[0]}", flush=True)
                        return {"executed": False,
                                "error": f"prior: {r.get('action')} predicted inert here (p={p:.2f}); pick a different action",
                                "valid_actions": _to_model(_solver._engine_action_names(session.game)),
                                **session.timing_payload()}
                    keep = [s[2] for s in scored if s not in inert]
                    if keep:
                        vetoes_this_step[0] += 1
                        _RANK_STATS["batch_drops"] += len(inert); _RANK_STATS["vetoes"] += 1
                        print(f"thui-rank: BATCH-DROP {len(inert)}/{len(props)} inert (min p={min(s[3] for s in inert):.3f}) obs={ranker.observed}", flush=True)
                        arguments = dict(arguments); arguments["actions"] = keep
        except Exception as exc:  # the ranker must never break the harness path
            _RANK_STATS["wrapper_errors"] += 1
            print(f"thui-rank: wrapper error (pass-through): {type(exc).__name__}: {exc}", flush=True)
        payload = step_env(arguments)
        try:
            ranker.observe(grid, payload)
        except Exception as exc:
            _RANK_STATS["wrapper_errors"] += 1
            print(f"thui-rank: observe skipped: {type(exc).__name__}: {exc}", flush=True)
        return payload

    return _orig_analyze(self, state_path, action_count, *args, valid_actions=valid_actions, step_env=rec_step_env, **kwargs)


_ta.ToolAgent.analyze = _rank_analyze
assert _ta.ToolAgent.analyze is _rank_analyze, "thui-rank: analyze wrap did not land"
print(f"thui-rank-v0: ToolAgent.analyze wrapped (veto p<{_RANK_VETO_P} after {_RANK_MIN_OBS} obs, <= {_RANK_PER_STEP}/step; never issues an action)", flush=True)
# ======================================================================================


## 7. Run the benchmark

In a real competition rerun (`KAGGLE_IS_COMPETITION_RERUN`), wait for the Kaggle gateway and
play the **live competition Arcade**. Otherwise — an interactive "Save & Run" — play the
competition's **bundled environment files offline**, with no gateway required, so the notebook
runs end-to-end without a submission. Teardown commands run afterward even if the run raises.

In [ ]:
# Build the live competition game list from the gateway's available environments.
def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# Build the offline game list from the competition's bundled environment files.
def _offline_games(env_dir: str):
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    arcade = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError(f"No offline environments found under {env_dir}.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# The gateway can take a while to come up; poll until it answers.
def _wait_for_gateway(base_url: str, timeout_s: float = 600.0) -> None:
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")


# Print the run preamble and persist the launcher's git status for diagnostics.
print((BUNDLE_DIR / "preamble.txt").read_text())
(WORKING_DIR / "git_status.txt").write_text((BUNDLE_DIR / "git_status.txt").read_text())

# arc_agi reads RECORDINGS_DIR and ARC_API_KEY from env (ArcadeSpec carries neither); operation
# mode, environments dir, and base url are all passed explicitly via the spec, so no env is needed.
os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

if TRUE_SUBMISSION:
    # Real submission: play the live competition Arcade served by the Kaggle gateway.
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    # The gateway boots asynchronously; wait before swapping in its game list.
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    bm.games = _competition_games()
else:
    # Interactive run: play the bundled competition environments offline (no gateway).
    # The competition's environment files ship alongside the wheelhouse in the competition dataset.
    competition_env_files = str(Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels").parent / "environment_files")
    bm.games = _offline_games(competition_env_files)
    # thui-rank-v0 smoke: three games, at the REAL seam.
    _SMOKE = ('tr87', 'sk48', 'sc25')
    _n0 = len(bm.games)
    bm.games = [g for g in bm.games if any(g.env_name.startswith(h) for h in _SMOKE)]
    print(f"thui-rank-v0: smoke filter {_n0} -> {len(bm.games)} games", flush=True)
    assert len(bm.games) == 3, f"thui-rank-v0: expected 3 games, got {len(bm.games)}"
    bm.solver.max_runtime_s_per_game = 900.0

bm.n_passes = 1
bm.game_weights = None

# Outside a real submission, stop ~10 min before the wall-clock budget for a graceful exit.
soft_end = None
if not TRUE_SUBMISSION:
    budget = float(getattr(target, "max_runtime_s", 0.0) or 0.0)
    if budget > 0:
        soft_end = datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(seconds=budget - min(600.0, budget / 2))

# Play the benchmark; teardown commands run even if the run raises.
try:
    await bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=TRUE_SUBMISSION)
    if not TRUE_SUBMISSION:
        # An offline run isn't scored, but Kaggle still expects a submission.parquet output.
        import pandas as pd

        pd.DataFrame(
            [["1_0", "1", True, 1]],
            columns=["row_id", "game_id", "end_of_game", "score"],
        ).to_parquet(WORKING_DIR / "submission.parquet", index=False)
finally:
    for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
        print(f"taaf.kaggle: teardown command: {command}", flush=True)
        subprocess.run(command, shell=True, check=False, cwd=WORKING_DIR, env=_command_env())

## 8. Show the diagnostics

A non-submission run writes `diagnostics.html` to `/kaggle/working`; it is rendered inline below
(and downloadable from the working directory). You should be able to click around through the links.

In [ ]:
from html import escape

from IPython.display import HTML, display

diagnostics_html = WORKING_DIR / "diagnostics.html"
if diagnostics_html.is_file():
    # Isolate the full document in an iframe so its styles don't leak into the notebook.
    display(
        HTML(
            f'<iframe srcdoc="{escape(diagnostics_html.read_text(), quote=True)}" '
            'width="100%" height="900" style="border:0"></iframe>'
        )
    )
else:
    print("No diagnostics.html — minimal diagnostics (real submission) suppresses it.")